[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/02-python-for-data-science/pyds-eda-univariate.ipynb)

# EDA Part 1: Univariate Analysis

*AIBits Academy · Machine Learning End To End · Python For Data Science*

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

Exploratory Data Analysis begins one variable at a time. Univariate analysis describes each column's shape, centre, spread, skewness, and outliers — the first thing you do with any fresh dataset.

> **💡 A note on this section vs. "EDA & Charting Basics"**
>
> The earlier prerequisite page introduced *which chart answers which question*. This page is the hands-on companion: we build a realistic messy dataset, clean it, and run the actual code for univariate EDA end to end.

## Building a Realistic Dataset

We generate a synthetic 50-row employee table (fixed seed, so it's reproducible) with three numeric columns — deliberately seeded with a few outliers — and three categorical columns. We then inject duplicate rows and missing values to mimic real data, and save it to CSV.

In [ ]:
import numpy as np
import pandas as pd
import random

np.random.seed(42)
random.seed(42)
n = 50
departments = ['Sales', 'Marketing', 'HR', 'Finance', 'IT']
qualifications = ['High School', 'Bachelor', 'Master', 'PhD']
marital = ['Single', 'Married', 'Divorced', 'Widowed']

def normal_with_outliers(size, mean, std, low, high):
    data = np.random.normal(mean, std, size)
    med = np.median(data)
    data[:low] = med - 3 * std              # a couple of low outliers
    data[low:low + high] = med + 3 * std    # a few high outliers
    return data

df = pd.DataFrame({
    'Employee_ID': range(1000, 1000 + n),
    'Salary': normal_with_outliers(n, 50000, 10000, 2, 3).astype(int),
    'Age': normal_with_outliers(n, 35, 5, 2, 3).astype(int),
    'Performance': normal_with_outliers(n, 7, 1.5, 2, 3).astype(int),
    'Qualification': random.choices(qualifications, k=n),
    'Marital_Status': random.choices(marital, k=n),
    'Department': random.choices(departments, k=n),
})

# inject 5 duplicate rows and 15 random missing values
df = pd.concat([df, df.sample(n=5, random_state=42)], ignore_index=True)
for _ in range(15):
    idx = np.random.randint(0, df.shape[0])
    col = random.choice(df.columns.tolist())
    df.loc[idx, col] = np.nan

df.to_csv('Employee_Details.csv', index=False)
print("shape:", df.shape)
print("duplicate rows:", df.duplicated().sum())
print("missing values:", df.isnull().sum().sum())

## Cleaning Before Analysis

Before describing distributions, we remove duplicates, impute missing values (median for numbers, mode for categories), and cap extreme outliers using the **IQR rule** — anything beyond 1.5×IQR from the quartiles is clipped to the boundary. The cleaned table is saved for the next page.

In [ ]:
import numpy as np, pandas as pd

df = pd.read_csv('Employee_Details.csv')
df = df.drop_duplicates()

num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())    # median for numeric
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])   # mode for categorical

def cap_outliers(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return np.clip(s, q1 - 1.5 * iqr, q3 + 1.5 * iqr)

for c in ['Salary', 'Age']:
    df[c] = cap_outliers(df[c])

df.to_csv('Cleaned_Employee_Details.csv', index=False)
print("cleaned shape:", df.shape)

## Descriptive Statistics

`describe()` gives count, mean, std, min/max, and quartiles in one call. **Skewness** measures asymmetry (positive = long right tail); **kurtosis** measures "tailedness" (high = heavy tails / more outliers).

In [ ]:
print(df['Salary'].describe().round(1))
print("skewness:", round(df['Salary'].skew(), 4))
print("kurtosis:", round(df['Salary'].kurtosis(), 4))

Skewness near 0 and low kurtosis tell us Salary is roughly symmetric with no heavy tails — as expected, since the IQR capping tamed the injected outliers.

## Visualising One Numeric Variable

The two workhorse plots for a numeric column: a **histogram** (with a KDE curve overlay) shows the distribution's shape, while a **box plot** summarises the median, quartiles, and any remaining outliers.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
sns.histplot(df['Salary'], bins=15, kde=True, ax=ax[0])
ax[0].set_title('Histogram of Salary (with KDE)')
sns.boxplot(x=df['Salary'], ax=ax[1])
ax[1].set_title('Box Plot of Salary')
plt.tight_layout()
plt.show()

Salary is roughly bell-shaped; the box plot shows the median line and quartile box with whiskers.

## Visualising One Categorical Variable

For a categorical column, `value_counts()` gives the frequency table and a **count plot** shows it visually.

In [ ]:
print(df['Department'].value_counts())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

order = df['Department'].value_counts().index
sns.countplot(x='Department', data=df, order=order)
plt.title('Count of Employees by Department')
plt.tight_layout()
plt.show()

The five departments are fairly balanced, from 10 to 11 employees each.

> **✅ What You Can Now Do**
>
> You can build and clean a messy dataset, summarise a numeric column (describe, skewness, kurtosis), and visualise one variable at a time with histograms, box plots, and count plots. Next: relationships *between* variables — bivariate and multivariate analysis.

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Centre and spread of a column

For the `delivery_minutes` series, store its mean in `avg`, median in `med` and standard deviation (pandas default) in `spread`. Which is more resistant to the slow outlier (95)? Think about it before you open the solution.

In [ ]:
import pandas as pd
delivery_minutes = pd.Series([22, 25, 24, 27, 23, 26, 95])
avg = med = spread = None   # TODO


In [ ]:
try:
    check("mean", round(avg, 2) == 34.57)
    check("median", med == 25)
    check("std", round(spread, 2) == 26.7)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import pandas as pd
delivery_minutes = pd.Series([22, 25, 24, 27, 23, 26, 95])
avg = delivery_minutes.mean()
med = delivery_minutes.median()
spread = delivery_minutes.std()

```

The single 95 drags the mean up to 34.6 while the median stays at 25 — the median is robust to outliers.

</details>

### Exercise 2 · Medium · Flag outliers with the IQR rule

Write `iqr_outliers(s)` returning the values of a Series that lie below Q1 − 1.5·IQR or above Q3 + 1.5·IQR (use `quantile(0.25)` / `quantile(0.75)`), as a sorted list.

In [ ]:
def iqr_outliers(s):
    pass   # TODO


In [ ]:
try:
    import pandas as pd
    check("finds the slow delivery", iqr_outliers(pd.Series([22, 25, 24, 27, 23, 26, 95])) == [95])
    check("finds both tails", iqr_outliers(pd.Series([1, 50, 52, 51, 49, 53, 100])) == [1, 100])
    check("clean data has none", iqr_outliers(pd.Series([10, 11, 12, 13, 14])) == [])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
def iqr_outliers(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return sorted(s[(s < lo) | (s > hi)].tolist())

```

</details>

### Exercise 3 · Stretch · Describe the shape of a distribution

Write `shape_label(s)` that returns `"right-skewed"` if the skewness is above 0.5, `"left-skewed"` if below −0.5, otherwise `"roughly symmetric"`.

In [ ]:
def shape_label(s):
    pass   # TODO


In [ ]:
try:
    import numpy as np, pandas as pd
    rng = np.random.default_rng(0)
    check("income-like data is right-skewed", shape_label(pd.Series(rng.lognormal(10, 0.8, 500))) == "right-skewed")
    check("bell curve is symmetric", shape_label(pd.Series(rng.normal(50, 5, 500))) == "roughly symmetric")
    check("mirror image is left-skewed", shape_label(pd.Series(-rng.lognormal(10, 0.8, 500))) == "left-skewed")
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
def shape_label(s):
    sk = s.skew()
    if sk > 0.5:
        return "right-skewed"
    if sk < -0.5:
        return "left-skewed"
    return "roughly symmetric"

```

</details>

---
*Back to the course: **Machine Learning End To End → EDA Part 1: Univariate Analysis**.*